<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Quantize Cosmos3-Nano and Cosmos3-Super to FP8

This notebook converts the Cosmos3-Nano (8B) and Cosmos3-Super (32B) checkpoints to
static-scale **FP8** with NVIDIA TensorRT Model Optimizer. FP8 roughly halves the model's
memory footprint and speeds up inference on NVIDIA GPUs with FP8 tensor cores, while
preserving generation quality.

Nano and Super use the **same recipe on a different checkpoint**. We quantize Nano first,
then Super.

## 1. Prerequisites

Use a Linux machine with an NVIDIA GPU, model access on Hugging Face, and either
`uvx hf@latest auth login` or `HF_TOKEN` set. Quantizing the Super (32B) checkpoints
needs a GPU with enough memory to hold the model in bf16 plus the quantizers; Nano (8B)
is comfortable on a single 48 GB GPU.

You also need free disk for the Hugging Face checkpoint cache and for the FP8 output
(each output is roughly the size of the bf16 source). Point `HF_HOME` at a large volume
in the next step.

> **Headless servers:** if you see `libGL.so.1: cannot open shared object file` when the
> VAE loads, install the system graphics libraries:
>
> ```bash
> apt-get install -y libgl1 libglib2.0-0
> ```

## 2. Configure Paths and Environment

The defaults are relative to this `cosmos` checkout and use the CUDA 13 (or 12.8) Torch
backend depending on your system. Override any of these before running the cell:

```bash
export COSMOS3_QUANTIZE_VENV=/path/to/.venv-cosmos3-quantize
export COSMOS3_TORCH_BACKEND=cu130       # or cu128
export HF_HOME=/path/to/large/huggingface/cache
export OUTPUT_ROOT=/path/to/fp8/outputs
```

In [1]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('COSMOS_ROOT   :', COSMOS_ROOT)
print('QUANTIZE_ROOT :', QUANTIZE_ROOT)
print('OUTPUT_ROOT   :', OUTPUT_ROOT)
print('venv          :', COSMOS3_QUANTIZE_VENV)

COSMOS_ROOT   : /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork
QUANTIZE_ROOT : /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/cookbooks/cosmos3/quantization
OUTPUT_ROOT   : /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/cookbooks/cosmos3/quantization/outputs
venv          : /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/.venv-cosmos3-quantize


## 3. Install Dependencies

This creates a dedicated virtual environment with PyTorch, NVIDIA TensorRT Model
Optimizer (ModelOpt — the FP8 quantization engine), Diffusers (VAE + schedulers) and
Transformers (tokenizer), then registers a Jupyter kernel for it. Run this once.

In [ ]:
# %%bash
# set -euo pipefail

# if ! command -v uv >/dev/null 2>&1; then
#   echo 'uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/'
#   exit 1
# fi

# export UV_LINK_MODE="${UV_LINK_MODE:-copy}"
# uv venv "$COSMOS3_QUANTIZE_VENV" --python 3.13 --seed --managed-python --allow-existing
# source "$COSMOS3_QUANTIZE_VENV/bin/activate"
# uv pip install --torch-backend="$COSMOS3_TORCH_BACKEND" \
#   "diffusers" \
#   "cosmos-framework @ git+https://github.com/NVIDIA/cosmos-framework" \
#   "nvidia-modelopt[torch]" \
#   accelerate datasets huggingface_hub imageio imageio-ffmpeg ipykernel \
#   numpy pillow safetensors torch torchvision transformers \
#   loguru iopath multi-storage-client boto3 wandb qwen_vl_utils

# "$COSMOS3_QUANTIZE_VENV/bin/python" -m ipykernel install --user \
#   --name cosmos3-quantize \
#   --display-name "Cosmos3 Quantize (Python 3.13)"

# echo
# echo "Installed into: $COSMOS3_QUANTIZE_VENV"
# echo "Next: switch this notebook kernel to: Cosmos3 Quantize (Python 3.13)"

Using CPython 3.13.13
Creating virtual environment with seed packages at: /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/.venv-cosmos3-quantize
 + pip==26.2.1
Activate with: source /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/.venv-cosmos3-quantize/bin/activate.csh
Using Python 3.13.13 environment at: /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/.venv-cosmos3-quantize
   Updating https://github.com/NVIDIA/cosmos-framework (HEAD)
    Updated https://github.com/NVIDIA/cosmos-framework (fa1881878ab9b234583d89319d56c7903be5161d)
Resolved 156 packages in 6.32s
   Building cosmos-framework @ git+https://github.com/NVIDIA/cosmos-framework@fa1881878ab9b234583d89319d56c7903be5161d
 Downloaded pydantic-core
      Built cosmos-framework @ git+https://github.com/NVIDIA/cosmos-framework@fa1881878ab9b234583d89319d56c7903be5161d
 Downloaded torchvision
 Downloaded obstore
 Downloaded multi-storage-client
 Downloaded nvidia-cudnn-frontend
 Downloaded nvidia-modelopt
 Downloade

Installed kernelspec cosmos3-quantize in /home/wkutak/.local/share/jupyter/kernels/cosmos3-quantize

Installed into: /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/.venv-cosmos3-quantize
Next: switch this notebook kernel to: Cosmos3 Quantize (Python 3.13)


## 4. Select the Quantization Kernel

The install cell registers the `Cosmos3 Quantize (Python 3.13)` Jupyter kernel.

**Switch this notebook to that kernel**, then run the restore cell below before
continuing. It can take a moment for a new kernel to appear in the notebook interface.

In [2]:
# Run this cell immediately after switching to the Cosmos3 Quantize kernel.
# It restores the same paths and cache settings as the Configure cell above.
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('environment restored — OUTPUT_ROOT:', OUTPUT_ROOT)

environment restored — OUTPUT_ROOT: /home/scratch.wkutak_other/dev/cosmos-lab/cosmos-fork/cookbooks/cosmos3/quantization/outputs


## 5. Verify GPU and Python Environment

Confirm the kernel sees a GPU and the quantization engine imports cleanly.

In [3]:
import torch, modelopt
print('torch          :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
print('ModelOpt       :', modelopt.__version__)

torch          : 2.14.0+cu130
CUDA available : False
ModelOpt       : 0.46.0


## 6. Load the Quantization Toolkit

The FP8 recipe lives in `cosmos_framework.quantization`.
`quantize_fp8_checkpoint(...)` loads, calibrates and exports the model;
the inspection helper below summarizes the resulting checkpoint.

Framework exports keep FP8 weights and store the activation policy in `transformer/config.json`.
By default, vLLM-Omni uses A16 (16-bit activations) for first/last 3 diffusion steps and the generator reasoner; middle steps use FP8.
A16 activation computation does not turn the FP8 checkpoint into a BF16 baseline.

In [ ]:
import json

from cosmos_framework.quantization import (
    SAMPLER_VIDEO_BASE,
    SHAPE_VIDEO,
    SHAPE_VIDEO_DEMO,
    quantize_fp8_checkpoint,
)
from safetensors import safe_open

# DEMO=1 (default): one calibration prompt at a small shape, so a run takes minutes.
# Set DEMO=0 for the production shape and 8 calibration prompts (the shipped recipe).
DEMO = os.environ.get('DEMO', '1') == '1'
NUM_SAMPLES = 1 if DEMO else 8
print(f'DEMO={DEMO}  (NUM_SAMPLES={NUM_SAMPLES})')


def inspect_checkpoint(output_dir):
    """Print a short, human-readable summary of an FP8 checkpoint."""
    output_dir = Path(output_dir)
    tdir = output_dir / 'transformer'
    n_fp8 = n_scale = 0
    example = []
    for shard in sorted(tdir.glob('*.safetensors')):
        with safe_open(str(shard), framework='pt') as h:
            for k in h.keys():
                if k.endswith(('.input_scale', '.weight_scale')):
                    n_scale += 1
                    if k.endswith('.input_scale') and len(example) < 3:
                        example.append((k, float(h.get_tensor(k).reshape(-1)[0])))
                elif k.endswith('.weight') and h.get_slice(k).get_dtype() == 'F8_E4M3':
                    n_fp8 += 1
    qcfg = json.load(open(output_dir / 'hf_quant_config.json'))
    print(f'FP8 checkpoint: {output_dir}')
    print(f"  quant_algo   : {qcfg.get('quant_algo')}  (method={qcfg.get('quant_method')})")
    print(f'  FP8 weights  : {n_fp8}')
    print(f'  scale tensors: {n_scale}')
    for k, v in example:
        print(f'    e.g. {k} = {v:.6g}')

<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEMO=True  (NUM_SAMPLES=1)


<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/modelopt/torch/__init__.py:51: UserWarning: transformers 5.14.1 is not tested with current version of modelopt and may cause issues. Please install recommended version with `pip install -U nvidia-modelopt[hf]` if working with HF models.
  _warnings.warn(


## Cosmos3-Nano

Nano is the 8B checkpoint — the fastest to quantize and a good first run.

### Quantize

Load Nano, calibrate it, and write the FP8 checkpoint to `OUTPUT_ROOT/nano-fp8`. In DEMO
mode this uses one calibration prompt at a small shape; set `DEMO=0` for the full calibration run.

In [ ]:
model_name_or_path = os.environ.get('C3_NANO_DIR', "nvidia/Cosmos3-Nano")
output_dir = OUTPUT_ROOT / 'nano-fp8'

quantize_fp8_checkpoint(
    model_name_or_path=model_name_or_path, output_dir=output_dir,
    profile='t2v', sampler=SAMPLER_VIDEO_BASE, shape=SHAPE_VIDEO_DEMO if DEMO else SHAPE_VIDEO,
    num_samples=NUM_SAMPLES, calibration_behavior='framework',
)

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [7]:
inspect_checkpoint(OUTPUT_ROOT / 'nano-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/nano-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 504
  scale tensors: 1008
    e.g. layers.0.mlp.down_proj.input_scale = 0.015346
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00354004
    e.g. layers.0.mlp.up_proj.input_scale = 0.00354004


## Cosmos3-Super

Super is the 32B checkpoint. The call is identical to Nano — only the checkpoint changes.
This is the heaviest run in the notebook; make sure your GPU has enough memory.

### Quantize

In [ ]:
model_name_or_path = os.environ.get('C3_SUPER_DIR', "nvidia/Cosmos3-Super")
output_dir = OUTPUT_ROOT / 'super-fp8'

quantize_fp8_checkpoint(
    model_name_or_path=model_name_or_path, output_dir=output_dir,
    profile='t2v', sampler=SAMPLER_VIDEO_BASE, shape=SHAPE_VIDEO_DEMO if DEMO else SHAPE_VIDEO,
    num_samples=NUM_SAMPLES, calibration_behavior='framework',
)

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [9]:
inspect_checkpoint(OUTPUT_ROOT / 'super-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 896
  scale tensors: 1792
    e.g. layers.0.mlp.down_proj.input_scale = 0.0262277
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00244141
    e.g. layers.0.mlp.up_proj.input_scale = 0.00244141
